# 25.07 - CV consolidation

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Full CV recode attempt.

**Priority:** P0 — rebuild the essential image-classification pipeline from memory.

Today is a closed-book consolidation day. Recreate the smallest complete CV workflow by selecting and configuring the allowed library tools: split without leakage, normalize from training data, build a CNN, train it, and evaluate accuracy plus Macro-F1.

## Core Ideas

- **Track the tensor contract:** images use `[N, C, H, W]`, `float32`, and labels use `[N]`, `long`.
- **Use library tools deliberately:** prefer a tested API over reimplementing it when the API itself is not the learning target.
- **Separate splitting from sampling:** `stratify=labels` preserves class proportions and keeps train/validation disjoint; `WeightedRandomSampler` then changes only training exposure.
- **Oversample training, not validation:** inverse-frequency sampling retains every original training row and draws minority classes more often with replacement. Validation remains untouched so its metrics describe distinct observations.
- **Split before fitting preprocessing:** validation information must not influence training normalization.
- **Keep class indices consistent:** the final layer width, label values, and metric class order must agree.
- **Use the standard training state:** `model.train()` with gradients for optimization; `model.eval()` and `torch.no_grad()` for inference.
- **Measure more than accuracy:** Macro-F1 and per-class recall expose weak classes that overall accuracy can hide.
- **Make the pipeline reproducible:** fix seeds and hold the split constant when comparing experiments.

## Setup and Prepared Image Data

The cell below provides a deterministic, intentionally imbalanced three-class image dataset. The learner does not need to create fixtures. Each class has a simple bright pattern plus noise: vertical, horizontal, or diagonal. The unequal counts make stratification and training-only oversampling observable.

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

SEED = 25
torch.manual_seed(SEED)
np.random.seed(SEED)

IMAGE_SIZE = 16
CLASS_COUNTS = [48, 36, 24]
NUM_CLASSES = 3
CLASS_NAMES = ["vertical", "horizontal", "diagonal"]

labels = torch.cat([
    torch.full((count,), class_index, dtype=torch.long)
    for class_index, count in enumerate(CLASS_COUNTS)
])
images = torch.zeros(len(labels), 1, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)

for row_index, label in enumerate(labels.tolist()):
    image = torch.rand(1, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32) * 0.12
    offset = (row_index % 3) - 1
    if label == 0:
        center = IMAGE_SIZE // 2 + offset
        image[:, :, center - 1:center + 2] += 0.85
    elif label == 1:
        center = IMAGE_SIZE // 2 + offset
        image[:, center - 1:center + 2, :] += 0.85
    else:
        for pixel in range(IMAGE_SIZE):
            shifted = min(max(pixel + offset, 0), IMAGE_SIZE - 1)
            image[:, pixel, shifted] += 0.85
    images[row_index] = image.clamp(0.0, 1.0)

print("images:", images.shape, images.dtype, images.device)
print("labels:", labels.shape, labels.dtype, torch.bincount(labels).tolist())

## Exercise 25-A: Stratify, normalize, and oversample training

Use scikit-learn's `train_test_split(..., stratify=labels)` to partition every original row while preserving class proportions. Calculate normalization statistics from training images only, then apply them to both partitions. Finally, configure PyTorch's `WeightedRandomSampler` with inverse-frequency sample weights so a training epoch draws each class equally in expectation. Set the epoch length to `largest_class_count * number_of_classes`, which oversamples minority classes without altering validation.

**Return structure — `split_and_normalize(images, labels, val_fraction, seed)`:**

- Returns a `dict` with exactly eight keys.
- `"train_images"`: CPU `torch.Tensor`, shape `[N_train, C, H, W]`, dtype `torch.float32`, normalized with training statistics.
- `"train_labels"`: CPU `torch.Tensor`, shape `[N_train]`, dtype `torch.long`.
- `"val_images"`: CPU `torch.Tensor`, shape `[N_val, C, H, W]`, dtype `torch.float32`, normalized with the same training statistics.
- `"val_labels"`: CPU `torch.Tensor`, shape `[N_val]`, dtype `torch.long`.
- `"mean"` and `"std"`: scalar CPU `torch.Tensor` objects; `std` is strictly positive.
- `"train_indices"` and `"val_indices"`: disjoint CPU `torch.Tensor` objects, shapes `[N_train]` and `[N_val]`, dtype `torch.long`; values index the original input rows.
- Together, `"train_indices"` and `"val_indices"` contain every original row exactly once; both partitions preserve the source class proportions.

**Return structure â€” `make_balanced_sampler(labels, seed)`:**

- Returns a `torch.utils.data.WeightedRandomSampler` with `replacement=True`.
- Its weight at row `i` is the inverse frequency of `labels[i]`.
- It yields `largest_class_count * number_of_classes` integer training-row indices per epoch, balancing class exposure in expectation while keeping all original training rows available.

In [ ]:
# TODO 25-A
def split_and_normalize(images, labels, val_fraction=0.25, seed=25):
    # Validate the inputs, call train_test_split with stratify=labels,
    # and calculate normalization statistics from the training rows only.
    raise NotImplementedError("TODO 25-A")


def make_balanced_sampler(labels, seed=25):
    # Build inverse-frequency sample weights and return WeightedRandomSampler.
    # Use replacement and an epoch size of max_class_count * number_of_classes.
    raise NotImplementedError("TODO 25-A")


# Smoke check: run this after implementing the function above.
smoke_split = split_and_normalize(images, labels, val_fraction=0.25, seed=SEED)
print("train:", smoke_split["train_images"].shape, smoke_split["train_labels"].shape)
print("validation:", smoke_split["val_images"].shape, smoke_split["val_labels"].shape)
print("stratified train counts:", torch.bincount(smoke_split["train_labels"]).tolist())
print("stratified validation counts:", torch.bincount(smoke_split["val_labels"]).tolist())
smoke_sampler = make_balanced_sampler(smoke_split["train_labels"], seed=SEED)
sampled_rows = torch.as_tensor(list(smoke_sampler), dtype=torch.long)
sampled_labels = smoke_split["train_labels"][sampled_rows]
print("oversampled epoch counts:", torch.bincount(sampled_labels).tolist())
print("train mean/std:", smoke_split["train_images"].mean().item(), smoke_split["train_images"].std().item())

train_loader = DataLoader(
    TensorDataset(smoke_split["train_images"], smoke_split["train_labels"]),
    batch_size=18,
    sampler=make_balanced_sampler(smoke_split["train_labels"], seed=SEED),
)
val_loader = DataLoader(
    TensorDataset(smoke_split["val_images"], smoke_split["val_labels"]),
    batch_size=18,
    shuffle=False,
)

## Exercise 25-B: Rebuild a minimal CNN

Use two convolution blocks, spatial downsampling, adaptive pooling, and a linear classifier. Do not apply softmax before `CrossEntropyLoss`.

**Return structure — `MinimalCNN(num_classes)` and its call `model(batch)`:**

- Construction returns an `nn.Module` instance whose parameters are `torch.float32` by default.
- Calling it with a `torch.float32` tensor of shape `[B, 1, H, W]` returns logits as a `torch.Tensor` of shape `[B, num_classes]`.
- Output dtype and device match the model parameters; logits are raw, unnormalized scores.

In [ ]:
# TODO 25-B
class MinimalCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        raise NotImplementedError("TODO 25-B: define layers")

    def forward(self, batch):
        raise NotImplementedError("TODO 25-B: implement forward")


# Smoke check: run this after implementing the class above.
smoke_model = MinimalCNN(NUM_CLASSES)
smoke_logits = smoke_model(smoke_split["train_images"][:4])
print("logits:", smoke_logits.shape, smoke_logits.dtype, smoke_logits.device)

## Exercise 25-C: Write one training epoch

Move batches to the model device, clear gradients, compute loss, backpropagate, and update parameters. Average loss by sample count rather than by number of batches.

**Return structure — `train_one_epoch(model, loader, optimizer, criterion)`:**

- Returns one Python `float`: the non-negative mean training loss per sample across the loader.
- Side effects: puts `model` in training mode, computes gradients, and updates model parameters through `optimizer`.
- The returned value is finite when the inputs and model outputs are finite.

In [ ]:
# TODO 25-C
def train_one_epoch(model, loader, optimizer, criterion):
    raise NotImplementedError("TODO 25-C")


# Smoke check: run this after implementing the function above.
smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=0.02)
smoke_loss = train_one_epoch(smoke_model, train_loader, smoke_optimizer, nn.CrossEntropyLoss())
print("one-epoch loss:", smoke_loss)

## Exercise 25-D: Evaluate accuracy, Macro-F1, and recall

Run batched inference without gradients, then use scikit-learn's integrated metric functions. Convert the CPU label tensors to NumPy arrays at the library boundary, pass an explicit class order, and set `zero_division=0` so absent predictions have defined scores.

**Return structure — `evaluate_model(model, loader, num_classes)`:**

- Returns a `dict` with exactly seven keys.
- `"logits"`: CPU `torch.Tensor`, shape `[N, num_classes]`, dtype `torch.float32`.
- `"labels"` and `"predictions"`: CPU `torch.Tensor` objects, shape `[N]`, dtype `torch.long`.
- `"confusion_matrix"`: CPU `torch.Tensor`, shape `[num_classes, num_classes]`, dtype `torch.long`; rows are truth and columns are predictions.
- `"per_class_recall"`: `list[float]` of length `num_classes` in class-index order.
- `"accuracy"` and `"macro_f1"`: Python `float` values in `[0.0, 1.0]`.

In [ ]:
# TODO 25-D
def evaluate_model(model, loader, num_classes):
    # Collect logits and labels with PyTorch, then call accuracy_score,
    # confusion_matrix, recall_score(average=None), and f1_score(average="macro").
    raise NotImplementedError("TODO 25-D")


# Smoke check: run this after implementing the function above.
smoke_metrics = evaluate_model(smoke_model, val_loader, NUM_CLASSES)
print("accuracy:", smoke_metrics["accuracy"])
print("Macro-F1:", smoke_metrics["macro_f1"])
print("confusion matrix:\n", smoke_metrics["confusion_matrix"])

## Exercise 25-E: Assemble the full recode experiment

Connect every earlier component into one reproducible pipeline. Re-seed before model construction, keep a loss history, and return the trained artifacts needed for review.

**Return structure — `run_recode_experiment(images, labels, epochs, batch_size, seed)`:**

- Returns a `dict` with exactly four keys.
- `"model"`: the trained `MinimalCNN` instance on CPU.
- `"split"`: the eight-key dictionary returned by `split_and_normalize`.
- `"history"`: `list[float]` of length `epochs`, containing finite non-negative mean training losses.
- `"metrics"`: the seven-key dictionary returned by `evaluate_model` for the validation partition.

In [ ]:
# TODO 25-E
def run_recode_experiment(images, labels, epochs=4, batch_size=18, seed=25):
    raise NotImplementedError("TODO 25-E")


# Smoke check: run this after implementing the function above.
smoke_run = run_recode_experiment(images, labels, epochs=4, batch_size=18, seed=SEED)
print("loss history:", smoke_run["history"])
print("validation Macro-F1:", smoke_run["metrics"]["macro_f1"])

## Test Cases

Run this cell after completing all TODO cells. A correct implementation should print `Day 25 tests passed`.

**Return structure — `run_day25_tests()`:**

- Returns `None`.
- Success is communicated by completing all assertions and printing exactly `Day 25 tests passed`.

In [ ]:
def run_day25_tests():
    assert "split_and_normalize" in globals(), "Missing function: split_and_normalize"
    assert "make_balanced_sampler" in globals(), "Missing function: make_balanced_sampler"
    assert "MinimalCNN" in globals(), "Missing class: MinimalCNN"
    assert "train_one_epoch" in globals(), "Missing function: train_one_epoch"
    assert "evaluate_model" in globals(), "Missing function: evaluate_model"
    assert "run_recode_experiment" in globals(), "Missing function: run_recode_experiment"

    split = split_and_normalize(images, labels, val_fraction=0.25, seed=SEED)
    assert set(split) == {"train_images", "train_labels", "val_images", "val_labels", "mean", "std", "train_indices", "val_indices"}
    assert split["train_images"].ndim == 4 and split["val_images"].ndim == 4
    assert split["train_images"].dtype == torch.float32
    assert split["train_labels"].dtype == torch.long
    assert split["train_images"].device.type == "cpu"
    train_counts = torch.bincount(split["train_labels"], minlength=NUM_CLASSES)
    val_counts = torch.bincount(split["val_labels"], minlength=NUM_CLASSES)
    source_counts = torch.bincount(labels, minlength=NUM_CLASSES)
    assert torch.equal(train_counts + val_counts, source_counts)
    assert torch.allclose(train_counts.float() / train_counts.sum(), source_counts.float() / source_counts.sum())
    assert torch.allclose(val_counts.float() / val_counts.sum(), source_counts.float() / source_counts.sum())
    assert len(split["train_labels"]) + len(split["val_labels"]) == len(labels)
    assert set(split["train_indices"].tolist()).isdisjoint(split["val_indices"].tolist())
    assert set(split["train_indices"].tolist()) | set(split["val_indices"].tolist()) == set(range(len(labels)))
    repeated_split = split_and_normalize(images, labels, val_fraction=0.25, seed=SEED)
    assert torch.equal(split["train_indices"], repeated_split["train_indices"])
    assert torch.equal(split["val_indices"], repeated_split["val_indices"])
    assert set(split["train_labels"].tolist()) == set(range(NUM_CLASSES))
    assert set(split["val_labels"].tolist()) == set(range(NUM_CLASSES))
    assert abs(split["train_images"].mean().item()) < 1e-5
    assert abs(split["train_images"].std().item() - 1.0) < 1e-4
    assert split["std"].item() > 0.0

    sampler = make_balanced_sampler(split["train_labels"], seed=SEED)
    assert isinstance(sampler, WeightedRandomSampler)
    assert sampler.replacement is True
    assert sampler.num_samples == int(train_counts.max().item()) * NUM_CLASSES
    expected_weights = train_counts.float().reciprocal()[split["train_labels"]]
    assert torch.allclose(sampler.weights.float(), expected_weights)

    model = MinimalCNN(NUM_CLASSES)
    logits = model(split["train_images"][:5])
    assert logits.shape == (5, NUM_CLASSES)
    assert logits.dtype == torch.float32 and logits.device.type == "cpu"

    loader = DataLoader(TensorDataset(split["train_images"], split["train_labels"]), batch_size=16)
    before = [parameter.detach().clone() for parameter in model.parameters()]
    loss = train_one_epoch(model, loader, torch.optim.Adam(model.parameters(), lr=0.02), nn.CrossEntropyLoss())
    assert isinstance(loss, float) and np.isfinite(loss) and loss >= 0.0
    assert any(not torch.equal(old, new.detach()) for old, new in zip(before, model.parameters()))

    val = DataLoader(TensorDataset(split["val_images"], split["val_labels"]), batch_size=16)
    metrics = evaluate_model(model, val, NUM_CLASSES)
    assert set(metrics) == {"logits", "labels", "predictions", "confusion_matrix", "per_class_recall", "accuracy", "macro_f1"}
    assert metrics["logits"].shape == (len(split["val_labels"]), NUM_CLASSES)
    assert metrics["labels"].dtype == torch.long and metrics["predictions"].dtype == torch.long
    assert metrics["confusion_matrix"].shape == (NUM_CLASSES, NUM_CLASSES)
    assert metrics["confusion_matrix"].dtype == torch.long
    assert int(metrics["confusion_matrix"].sum()) == len(split["val_labels"])
    assert len(metrics["per_class_recall"]) == NUM_CLASSES
    expected_f1 = f1_score(
        metrics["labels"].numpy(),
        metrics["predictions"].numpy(),
        labels=np.arange(NUM_CLASSES),
        average="macro",
        zero_division=0,
    )
    assert np.isclose(metrics["macro_f1"], expected_f1)
    assert 0.0 <= metrics["accuracy"] <= 1.0
    assert 0.0 <= metrics["macro_f1"] <= 1.0

    result = run_recode_experiment(images, labels, epochs=4, batch_size=18, seed=SEED)
    assert set(result) == {"model", "split", "history", "metrics"}
    assert isinstance(result["model"], MinimalCNN)
    assert len(result["history"]) == 4
    assert all(isinstance(value, float) and np.isfinite(value) and value >= 0.0 for value in result["history"])
    assert result["history"][-1] < result["history"][0]
    assert result["metrics"]["macro_f1"] >= 0.90

    print("Day 25 tests passed")


run_day25_tests()

## Day 25 Checklist

- [ ] I can state every important tensor shape, dtype, and device.
- [ ] I can explain why stratification and training-time oversampling solve different problems.
- [ ] I can configure `train_test_split` and `WeightedRandomSampler` without altering validation.
- [ ] I split data before calculating preprocessing statistics.
- [ ] I can rebuild a small CNN without copying an old implementation.
- [ ] I switch correctly between training and evaluation modes.
- [ ] I use scikit-learn to calculate accuracy, confusion matrix, per-class recall, and Macro-F1.
- [ ] I can rerun the experiment with the same seed and explain the result.
- [ ] I recorded the parts I could not recode from memory for targeted review.